# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the FAIR^2 dataset package using the `mlcroissant` library.

### Dataset Source

The dataset source is described via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

> **Note:** This dataset describes the results of mass spectrometry analyses on human extracellular vesicles and includes protein identification and quantification information across several samples. All data, record sets, fields, and columns are referenced by their Croissant `@id` values.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load the dataset and its metadata using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\nDescription: {metadata.description}\nPublished: {metadata.date_published}")

## 2. Data Overview

List available record sets and their associated field `@id`s. All further data referencing will use these `@id` values.

> **Note:** The actual `@id` values may differ or be updated. We enumerate record sets and fields programmatically.

In [ ]:
# List all available record sets and their field @id values

print("Available record sets and fields:")
record_sets = list(dataset.record_sets)
record_set_ids = []
for record_set in record_sets:
    print(f"- Record set: {record_set['@id']} (name: {record_set.get('name', record_set['@id'])})")
    fields = record_set.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    if not fields:
        print("  [No fields found in this record set]")
    else:
        for field in fields:
            field_id = field['@id'] if isinstance(field, dict) else field
            print(f"    - Field @id: {field_id}")
    record_set_ids.append(record_set['@id'])
if not record_set_ids:
    print("No record sets found. Please check the dataset's Croissant schema.")

## 3. Data Extraction

Load data from a selected record set into a DataFrame for analysis. Use the record set `@id` and field `@id`s from the overview above.

In [ ]:
# Select one or more record sets by @id for further analysis.
# Here, the first record set found is used. Replace the selection as needed for your analysis.

if record_set_ids:
    record_sets_to_load = [record_set_ids[0]]
    dataframes = {}
    for rs_id in record_sets_to_load:
        print(f"Loading records from record set @id: {rs_id}")
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print("Fields (columns) in this record set:")
        print(df.columns.tolist())
        display(df.head())
else:
    print("No record sets available to load records.")

## 4. Exploratory Data Analysis (EDA)

Analyze and process data using the loaded DataFrame. The following are typical EDA steps—filtering, normalization, and grouping—adapted to work with `@id` field names.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Use the first available record set for EDA
if record_set_ids:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    print(f"Using record set: {record_set_id}\nAvailable columns: {df.columns.tolist()}")

    # Select a numeric field @id for demonstration. Replace this if you know the field you want to analyze.
    numeric_field_candidates = df.select_dtypes(include=np.number).columns.tolist()
    if not numeric_field_candidates:
        print("No numeric fields found for EDA.")
    else:
        numeric_field_id = numeric_field_candidates[0]  # For example
        print(f"Selected numeric field for filtering/normalization: {numeric_field_id}")

        # Filtering records with field value > threshold
        threshold = np.nanmean(df[numeric_field_id]) if np.nanmean(df[numeric_field_id]) else 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (first 5 displayed):")
        display(filtered_df.head())

        # Normalizing the selected numeric field
        filtered_df = filtered_df.copy()
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records (first 5 displayed):")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping: Use the first available categorical field
        group_field_candidates = df.select_dtypes(include=['object', 'category']).columns.tolist()
        group_field_id = None
        for col in group_field_candidates:
            if col != numeric_field_id:
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
            print(f"Grouped (mean) {numeric_field_id} by {group_field_id} (first 5 groups):")
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
else:
    print("No record set or DataFrame loaded for EDA.")

## 5. Visualization

Visualize data distributions or relationships between fields. This example shows a histogram and a scatter plot for two numeric fields using their `@id` values.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# If there are at least one or two numeric columns, plot histogram and scatter
if record_set_ids:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    if len(numeric_cols) > 0:
        plt.figure(figsize=(6, 4))
        sns.histplot(df[numeric_cols[0]].dropna(), kde=True)
        plt.title(f'Histogram of field {numeric_cols[0]}')
        plt.xlabel(numeric_cols[0])
        plt.show()
    if len(numeric_cols) > 1:
        plt.figure(figsize=(6, 4))
        sns.scatterplot(x=df[numeric_cols[0]], y=df[numeric_cols[1]])
        plt.title(f'Scatterplot: {numeric_cols[0]} vs. {numeric_cols[1]}')
        plt.xlabel(numeric_cols[0])
        plt.ylabel(numeric_cols[1])
        plt.show()
    else:
        print("Not enough numeric fields for scatter plot.")
else:
    print("No data available for visualization.")

## 6. Conclusion

- The FAIR^2 dataset describing mass spectrometry analysis of extracellular vesicles has been loaded and explored using the `mlcroissant` library.
- Record sets and fields were referenced by their Croissant `@id` values as per the FAIR and Croissant conventions.
- Preliminary EDA and visualizations were performed. For more targeted analyses (e.g., specific proteins or samples), use the field `@id`s discovered in section 2 to select columns of interest.

> **Tip:** For more advanced analyses, further reference the Croissant specification and schema to leverage relationships between datasets, record sets, and other linked resources.